# atmaCup #15 1位風 5 パターン（割り算 + Implicit）

**ベース**: 0.75493 の土台（55 特徴: embedding + PCA16 + doc_x_critic_te）。

**5 パターン**:
| # | 内容 | 提出ファイル |
|---|------|-------------|
| 1 | 割り算: genre_Documentary / 批評家平均 | submission_atmacup_ratio_genre_doc.csv |
| 2 | 割り算: critic_name_te_ts / 批評家平均 | submission_atmacup_ratio_critic_te.csv |
| 3 | Implicit ALS factors=16 | submission_atmacup_implicit_als16.csv |
| 4 | Implicit BPR factors=16 | submission_atmacup_implicit_bpr16.csv |
| 5 | 割り算: runtime_bin / 批評家平均 | submission_atmacup_ratio_runtime_bin.csv |

**依存**: `pip install implicit`（requirements.txt に追加済み）。

## 1. セットアップ（最高精度ベース 55 特徴）

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from lib.improvement_candidates import (
    get_setup,
    run_atmacup_ratio,
    run_atmacup_implicit,
)

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)

print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")
print(f"提出先: {ctx.submissions_dir}")

Train: 653,507, Test: 40,716, Features: 55
提出先: outputs/submissions


## 2. 5 パターン実行・提出 CSV 保存

In [2]:
def _report(name, r):
    if r.get("path"):
        p = r["path"]
        name_part = p.name if hasattr(p, "name") else Path(p).name
        print(f"  [{name}] → {name_part}  ({'OK' if r.get('ok') else r.get('message', '')})")
    else:
        print(f"  [{name}] スキップ: {r.get('message', '')}")

results = []

r1 = run_atmacup_ratio(ctx, "genre_Documentary", "genre_doc")
_report("1 割り算(genre_Documentary)", r1)
results.append(r1)

r2 = run_atmacup_ratio(ctx, "critic_name_te_ts", "critic_te")
_report("2 割り算(critic_name_te_ts)", r2)
results.append(r2)

r3 = run_atmacup_implicit(ctx, "als", 16, "als16")
_report("3 Implicit ALS 16", r3)
results.append(r3)

r4 = run_atmacup_implicit(ctx, "bpr", 16, "bpr16")
_report("4 Implicit BPR 16", r4)
results.append(r4)

r5 = run_atmacup_ratio(ctx, "runtime_bin", "runtime_bin")
_report("5 割り算(runtime_bin)", r5)
results.append(r5)

print(f"\n→ {sum(1 for r in results if r.get('ok'))} / 5 本 OK")

  [1 割り算(genre_Documentary)] → submission_atmacup_ratio_genre_doc.csv  (OK)
  [2 割り算(critic_name_te_ts)] → submission_atmacup_ratio_critic_te.csv  (OK)


  0%|          | 0/15 [00:00<?, ?it/s]

  [3 Implicit ALS 16] → submission_atmacup_implicit_als16.csv  (OK)


  0%|          | 0/100 [00:00<?, ?it/s]

ValueError: could not broadcast input array from shape (17,) into shape (16,)

## 3. 4 と 5 だけ実行（1〜3 はスキップ）

1〜3 がすでに完了しているとき、4（BPR）と 5（割り算 runtime_bin）だけ回す用。**セットアップのセル（上）を実行したあと、このセルだけ実行すればよい。**

In [2]:
def _report(name, r):
    if r.get("path"):
        p = r["path"]
        name_part = p.name if hasattr(p, "name") else Path(p).name
        print(f"  [{name}] → {name_part}  ({'OK' if r.get('ok') else r.get('message', '')})")
    else:
        print(f"  [{name}] スキップ: {r.get('message', '')}")

r4 = run_atmacup_implicit(ctx, "bpr", 16, "bpr16")
_report("4 Implicit BPR 16", r4)

r5 = run_atmacup_ratio(ctx, "runtime_bin", "runtime_bin")
_report("5 割り算(runtime_bin)", r5)

  0%|          | 0/100 [00:00<?, ?it/s]

  [4 Implicit BPR 16] → submission_atmacup_implicit_bpr16.csv  (OK)
  [5 割り算(runtime_bin)] → submission_atmacup_ratio_runtime_bin.csv  (OK)
